<a href="https://colab.research.google.com/github/janumpallydeepthi/Infosys_FranciseOps_AI/blob/main/working_model_milestone1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly

In [ ]:
from google.colab import userdata

try:
    print("JWT_SECRET      :", "✅" if userdata.get('JWT_SECRET') else "❌ empty")
    print("EMAIL_ADDRESS   :", "✅" if userdata.get('EMAIL_ADDRESS') else "❌ empty")
    print("EMAIL_PASSWORD  :", "✅" if userdata.get('EMAIL_PASSWORD') else "❌ empty")
    print("NGROK_AUTHTOKEN :", "✅" if userdata.get('NGROK_AUTHTOKEN') else "❌ empty")
except Exception as e:
    print("❌ Error reading secrets:", e)

In [ ]:
# import os
# if os.path.exists("infosys_portal.db"):
#     os.remove("infosys_portal.db")
#     print("Database deleted. Restart the app to recreate with new admin.")
# else:
#     print("Database not found – nothing to delete.")

In [ ]:
%%writefile app.py

import os, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib, streamlit as st
import plotly.graph_objects as go
from streamlit_option_menu import option_menu
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from google.colab import userdata
import re

# ─── SECRETS ──────────────────────────────────────────────────
JWT_SECRET = os.environ.get('JWT_SECRET')
EMAIL_ADDRESS = os.environ.get('EMAIL_ADDRESS')
EMAIL_PASSWORD = os.environ.get('EMAIL_PASSWORD')

if not JWT_SECRET or not EMAIL_ADDRESS or not EMAIL_PASSWORD:
    st.error("❌ Missing secrets! Please set JWT_SECRET, EMAIL_ADDRESS, and EMAIL_PASSWORD in Colab Secrets.")
    st.stop()

OTP_EXPIRY_MINUTES = 5

# ─── Styling ─────────────────────────────────────────────
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#ffd803"\nbackgroundColor="#f9fcfc"\nsecondaryBackgroundColor="#e3f6f5"\ntextColor="#2d334a"\n')

st.set_page_config(page_title="Infosys Franchise Analytics & Management", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#f9fcfc", "bg_sidebar": "#e3f6f5", "bg_card": "#ffffff", "bg_card_alt": "#bae8e8",
    "text_main": "#2d334a", "text_heading": "#272343", "text_muted": "#64748b",
    "accent": "#ffd803", "accent_hover": "#e6c300", "accent_text": "#272343",
    "border": "#272343", "border_light": "#bae8e8", "success": "#34d399", "danger": "#f87171"
}

st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');
    html, body, .stApp {{ background: {COLORS['bg_main']} !important; font-family: 'Inter', sans-serif !important; color: {COLORS['text_main']} !important; }}
    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}
    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background-color: {COLORS['accent']} !important; border: 2px solid {COLORS['border']} !important;
        border-radius: 8px !important; padding: 6px !important; margin: 8px !important;
        box-shadow: 3px 3px 0px {COLORS['border']} !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: {COLORS['text_heading']} !important; color: {COLORS['text_heading']} !important; stroke: {COLORS['text_heading']} !important;
    }}
    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_heading']} !important; }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_heading']} !important; }}
    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"] {{ background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border']} !important; border-radius: 10px !important; }}
    div[data-baseweb="input"]:focus-within {{ border-color: {COLORS['accent']} !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; }}
    input, div[data-baseweb="select"] span {{ color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important; }}
    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important; color: {COLORS['accent_text']} !important;
        border: 2px solid {COLORS['border']} !important; border-radius: 10px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 700 !important; font-size: 14px !important;
        height: 48px !important; min-height: 48px !important; white-space: nowrap !important;
        display: flex !important; align-items: center !important; justify-content: center !important;
        padding: 0px 16px !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['accent_hover']} !important; transform: translate(-2px, -2px) !important;
        box-shadow: 6px 6px 0px {COLORS['border']} !important;
    }}
    section[data-testid="stSidebar"] {{ background: {COLORS['bg_sidebar']} !important; border-right: 2px solid {COLORS['border']} !important; }}
    .pn-card {{ background: {COLORS['bg_card']}; border: 2px solid {COLORS['border']}; border-radius: 14px; padding: 24px; box-shadow: 4px 4px 0px {COLORS['border_light']}; }}
</style>
""", unsafe_allow_html=True)

# ─── VALIDATION ──────────────────────────────────────────────
def is_valid_email(email):
    if '@' not in email:
        return False
    local, rest = email.split('@', 1)
    if '.' not in rest:
        return False
    first_dot = rest.find('.')
    domain_part = rest[:first_dot]
    last_dot = rest.rfind('.')
    tld_part = rest[last_dot + 1:]
    if len(re.findall(r'[A-Za-z]', local)) < 2:
        return False
    if len(re.findall(r'[A-Za-z]', domain_part)) < 2:
        return False
    if len(re.findall(r'[A-Za-z]', tld_part)) < 2:
        return False
    return True

def is_valid_password(password):
    if len(password) < 8:
        return False
    if not re.search(r'[A-Z]', password):
        return False
    if not re.search(r'[a-z]', password):
        return False
    if not re.search(r'[0-9]', password):
        return False
    if not re.search(r'[!@#$%^&*(),.?":{}|<>]', password):
        return False
    return True

# ─── DATABASE ─────────────────────────────────────────────────
def get_db():
    return sqlite3.connect("infosys_portal.db", check_same_thread=False)

def hash_txt(t):
    return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()

def check_txt(t, h):
    return bcrypt.checkpw(t.encode(), h.encode()) if h else False

# ─── INIT DATABASE & ADMIN ──────────────────────────────────
with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE,
        email TEXT UNIQUE,
        password_hash TEXT,
        security_question TEXT,
        security_answer_hash TEXT
    )""")
    conn.execute("""CREATE TABLE IF NOT EXISTS password_history (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        email TEXT,
        password_hash TEXT,
        created_at TEXT,
        FOREIGN KEY(email) REFERENCES users(email)
    )""")
    if not conn.execute("SELECT id FROM users WHERE email='franchise.admin@infosys.com'").fetchone():
        admin_hash = hash_txt("admin@123")
        conn.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                     ("Administrator", "franchise.admin@infosys.com", admin_hash,
                      "What is your pet name?", hash_txt("admin")))
        conn.execute("INSERT INTO password_history (email, password_hash, created_at) VALUES (?, ?, ?)",
                     ("franchise.admin@infosys.com", admin_hash, datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
    conn.commit()

# ─── JWT & OTP ────────────────────────────────────────────────
def make_jwt(email):
    return jwt.encode(
        {"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)},
        JWT_SECRET,
        algorithm="HS256"
    )

def verify_jwt(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except:
        return None

def generate_otp():
    return f"{secrets.randbelow(900000) + 100000}"

def make_otp_token(email, otp):
    payload = {
        "sub": email,
        "otp_hash": hash_txt(otp),
        "type": "password_reset_otp",
        "iat": datetime.datetime.utcnow(),
        "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != email or payload.get("type") != "password_reset_otp":
            return False, "Security token mismatch."
        if check_txt(input_otp, payload["otp_hash"]):
            return True, "Valid"
        return False, "Invalid 6-digit OTP code."
    except jwt.ExpiredSignatureError:
        return False, f"⚠️ This OTP code expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new one."
    except Exception:
        return False, "Invalid or corrupted verification token."

def send_professional_email(to_email, otp, app_pass):
    msg = MIMEMultipart('alternative')
    msg['From'] = f"Infosys Franchise Support <{EMAIL_ADDRESS}>"
    msg['To'] = to_email
    msg['Subject'] = "Infosys Franchise Analytics & Management - Verification Code"
    msg['Date'] = formatdate(localtime=True)
    msg['Message-ID'] = make_msgid()
    msg['Reply-To'] = EMAIL_ADDRESS

    text_body = f"Your verification code for Infosys Franchise Analytics & Management is: {otp}\nThis code will expire in {OTP_EXPIRY_MINUTES} minutes.\nIf you did not request this code, please ignore this email."

    html_body = f"""
    <!DOCTYPE html>
    <html>
    <head><style>
        body {{ font-family: Arial, sans-serif; background-color: #f9fcfc; margin: 0; padding: 20px; }}
        .container {{ max-width: 500px; margin: 0 auto; background-color: #ffffff; border: 2px solid #272343; border-radius: 12px; padding: 30px; text-align: center; }}
        .title {{ color: #272343; font-size: 20px; font-weight: bold; margin-bottom: 15px; }}
        .otp-box {{ background-color: #ffd803; color: #272343; font-size: 28px; font-weight: bold; letter-spacing: 5px; padding: 15px 20px; border: 2px solid #272343; border-radius: 8px; display: inline-block; margin: 10px 0; }}
        .footer {{ color: #718096; font-size: 12px; margin-top: 25px; border-top: 1px solid #edf2f7; padding-top: 15px; }}
    </style></head>
    <body>
        <div class="container">
            <div class="title">Infosys Franchise Analytics & Management</div>
            <div class="text">We received a request to reset your password for <b>{to_email}</b>. Please use the verification code below:</div>
            <div class="otp-box">{otp}</div>
            <div class="text">This code expires in <b>{OTP_EXPIRY_MINUTES} minutes</b>.</div>
            <div class="footer">If you did not request this code, you can safely ignore this email.<br>&copy; 2026 Infosys Franchise Analytics & Management.</div>
        </div>
    </body>
    </html>
    """
    msg.attach(MIMEText(text_body, 'plain'))
    msg.attach(MIMEText(html_body, 'html'))
    try:
        s = smtplib.SMTP('smtp.gmail.com', 587)
        s.starttls()
        s.login(EMAIL_ADDRESS, app_pass if app_pass else "")
        s.sendmail(EMAIL_ADDRESS, to_email, msg.as_string())
        s.quit()
        return True, "Email sent successfully!"
    except Exception as e:
        return False, f"SMTP Error: {str(e)}"

# ─── SESSION STATE ───────────────────────────────────────────
for k, v in [
    ("token", None),
    ("page", "Login"),
    ("reset_email", None),
    ("reset_mode", None),
    ("otp_stage", None),
    ("otp_jwt", None),
    ("sq_p", None),
    ("admin_logged", False)
]:
    if k not in st.session_state:
        st.session_state[k] = v

def navigate(p):
    st.session_state.page = p
    st.rerun()

def auth_header(title, sub="Franchise Analytics & Management"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">Infosys Franchise Analytics & Management</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.1rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

# ─── ADMIN LOGIN (without sign up) ──────────────────────────────────
ADMIN_USERNAME = "admin"
ADMIN_PASSWORD = "admin@123"

def admin_login_page():
    auth_header("Admin Login", "Secure Administrator Access")
    with st.form("admin_login_form"):
        uname = st.text_input("Admin Username")
        pwd = st.text_input("Admin Password", type="password")
        submitted = st.form_submit_button("Login as Admin →", use_container_width=True)
        if submitted:
            if not uname or not pwd:
                st.error("⚠️ Both fields are mandatory.")
            elif uname == ADMIN_USERNAME and pwd == ADMIN_PASSWORD:
                st.session_state.admin_logged = True
                st.success("✅ Admin login successful!")
                time.sleep(0.5)
                st.rerun()
            else:
                st.error("❌ Invalid admin credentials.")
    if st.button("← Back to User Login", use_container_width=True):
        navigate("Login")

def admin_dashboard():
    st.markdown(f"""
    <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
        <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Franchise Analytics &amp; Management</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Admin Panel</div></div>
        <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};white-space:nowrap;">🛡️ Admin</div>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("### 📋 Registered Users")
    with get_db() as c:
        users = c.execute("SELECT username, email FROM users ORDER BY id").fetchall()
    if users:
        data = [{"Username": u[0], "Email": u[1]} for u in users]
        st.table(data)
    else:
        st.info("No users registered yet.")

    if st.button("Admin Logout", use_container_width=True):
        st.session_state.admin_logged = False
        st.rerun()

# MAIN ROUTING
if st.session_state.admin_logged:
    admin_dashboard()
else:
    if not st.session_state.token:
        if st.session_state.page not in ["Login", "Signup", "Forgot", "AdminLogin"]:
            st.session_state.page = "Login"

        _, mid, _ = st.columns([1, 1.45, 1])
        with mid:
            if st.session_state.page == "Login":
                auth_header("Sign in to your account")
                email = st.text_input("Email address", placeholder="you@franchise.com").lower().strip()
                pwd = st.text_input("Password", type="password", placeholder="••••••••")
                st.markdown("<br>", unsafe_allow_html=True)

                col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
                if col_l.button("Sign In →", use_container_width=True):
                    if not email or not pwd:
                        st.error("⚠️ Both fields are mandatory.")
                    else:
                        with get_db() as c:
                            r = c.execute("SELECT password_hash FROM users WHERE email=?", (email,)).fetchone()
                        if r and check_txt(pwd, r[0]):
                            st.session_state.token = make_jwt(email)
                            navigate("Dashboard")
                        else:
                            st.error("❌ Invalid credentials.")
                if col_c.button("Create Account", use_container_width=True):
                    navigate("Signup")
                if col_r.button("Forgot Password", use_container_width=True):
                    navigate("Forgot")
                if st.button("Admin Login", use_container_width=True):
                    navigate("AdminLogin")

            elif st.session_state.page == "Signup":
                auth_header("Create an account", "Join Infosys Franchise today")
                with st.form("signup_form"):
                    uname = st.text_input("Full name / Username", placeholder="Jane Doe")
                    email = st.text_input("Email address", placeholder="you@franchise.com").lower().strip()
                    pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters").strip()
                    confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password").strip()
                    sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
                    sa = st.text_input("Your answer", placeholder="Security answer")
                    st.markdown("<br>", unsafe_allow_html=True)
                    submitted = st.form_submit_button("Create Account & Login →", use_container_width=True)

                    if submitted:
                        if not uname or not email or not pwd or not confirm_pwd or not sa:
                            st.error("⚠️ All fields are mandatory.")
                        elif not is_valid_email(email):
                            st.error("❌ Invalid email format. Must be at least 2 letters before @, 2 between @ and ., and 2 after .")
                        elif len(pwd) < 8:
                            st.error("⚠️ Password must be at least 8 characters long.")
                        elif not is_valid_password(pwd):
                            st.error("❌ Password must contain at least one uppercase, one lowercase, one digit, and one special symbol.")
                        elif pwd != confirm_pwd:
                            st.error("❌ Passwords do not match.")
                        else:
                            try:
                                with get_db() as c:
                                    hashed = hash_txt(pwd)
                                    c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                                              (uname, email, hashed, sq, hash_txt(sa.lower().strip())))
                                    c.execute("INSERT INTO password_history (email, password_hash, created_at) VALUES (?, ?, ?)",
                                              (email, hashed, datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
                                    c.commit()
                                st.session_state.token = make_jwt(email)
                                st.success("✅ Account created!")
                                time.sleep(1)
                                navigate("Dashboard")
                            except sqlite3.IntegrityError:
                                st.error("❌ Email or Username already registered.")
                if st.button("← Back to Sign In", use_container_width=True):
                    navigate("Login")

            elif st.session_state.page == "Forgot":
                auth_header("Reset your password", "Choose your verification method")

                if not st.session_state.reset_email:
                    email = st.text_input("Registered email address", placeholder="you@franchise.com").lower().strip()
                    method = st.radio("Select recovery method", ["Security Question", "OTP via Email"], horizontal=True)
                    st.markdown("<br>", unsafe_allow_html=True)

                    if st.button("Continue →", use_container_width=True):
                        if not email:
                            st.error("⚠️ Please enter your email.")
                        elif not is_valid_email(email):
                            st.error("❌ Invalid email format.")
                        else:
                            with get_db() as c:
                                user = c.execute("SELECT security_question FROM users WHERE email=?", (email,)).fetchone()
                            if not user:
                                st.error("❌ Email not registered.")
                            else:
                                st.session_state.reset_email = email
                                if method == "Security Question":
                                    st.session_state.reset_mode = "sq"
                                    st.session_state.sq_p = user[0]
                                    st.rerun()
                                else:
                                    st.session_state.reset_mode = "otp"
                                    otp = generate_otp()
                                    ok, msg = send_professional_email(email, otp, EMAIL_PASSWORD)
                                    if ok:
                                        st.session_state.otp_jwt = make_otp_token(email, otp)
                                        st.session_state.otp_stage = "otp"
                                        st.success("✅ 6-digit OTP sent to your email!")
                                        time.sleep(1)
                                        st.rerun()
                                    else:
                                        st.error(f"❌ {msg}")
                                        st.session_state.reset_email = None

                elif st.session_state.reset_mode == "sq":
                    st.info(f"❓ **Security Question:** {st.session_state.sq_p}")
                    with st.form("sq_reset_form"):
                        ans = st.text_input("Your answer").lower().strip()
                        npw = st.text_input("New password (min 8 chars)", type="password").strip()
                        confirm_npw = st.text_input("Confirm new password", type="password").strip()
                        st.markdown("<br>", unsafe_allow_html=True)
                        submitted = st.form_submit_button("Reset Password →", use_container_width=True)

                        if submitted:
                            if not ans or not npw or not confirm_npw:
                                st.error("⚠️ All fields are mandatory.")
                            elif len(npw) < 8:
                                st.error("⚠️ Password must be at least 8 characters long.")
                            elif not is_valid_password(npw):
                                st.error("❌ Password must contain at least one uppercase, one lowercase, one digit, and one special symbol.")
                            elif npw != confirm_npw:
                                st.error("❌ Passwords do not match.")
                            else:
                                with get_db() as c:
                                    r = c.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                                    if r and check_txt(ans, r[0]):
                                        history = c.execute("SELECT password_hash FROM password_history WHERE email=?", (st.session_state.reset_email,)).fetchall()
                                        if any(check_txt(npw, h[0]) for h in history):
                                            st.error("❌ You have used this password before. Please choose a different password.")
                                        else:
                                            new_hash = hash_txt(npw)
                                            c.execute("UPDATE users SET password_hash=? WHERE email=?", (new_hash, st.session_state.reset_email))
                                            c.execute("INSERT INTO password_history (email, password_hash, created_at) VALUES (?, ?, ?)",
                                                      (st.session_state.reset_email, new_hash, datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
                                            c.commit()
                                            # Verify
                                            verify_hash = c.execute("SELECT password_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()[0]
                                            if check_txt(npw, verify_hash):
                                                st.success("✅ Password updated successfully!")
                                                time.sleep(1)
                                                st.session_state.reset_email = None
                                                st.session_state.reset_mode = None
                                                st.session_state.sq_p = None
                                                navigate("Login")
                                            else:
                                                st.error("❌ Password update failed. Please try again.")
                                    else:
                                        st.error("❌ Incorrect security answer.")

                elif st.session_state.reset_mode == "otp":
                    if st.session_state.otp_stage == "otp":
                        st.info(f"OTP sent to **{st.session_state.reset_email}** (Valid for {OTP_EXPIRY_MINUTES} mins).")
                        with st.form("otp_verify_form"):
                            otp_input = st.text_input("6-Digit Verification Code", placeholder="e.g. 849201", max_chars=6)
                            st.markdown("<br>", unsafe_allow_html=True)
                            submitted = st.form_submit_button("Verify Code →", use_container_width=True)
                            if submitted:
                                if not otp_input or len(otp_input) != 6:
                                    st.error("⚠️ Please enter the valid 6-digit code.")
                                else:
                                    ok, msg = verify_otp_token(st.session_state.otp_jwt, otp_input, st.session_state.reset_email)
                                    if ok:
                                        st.session_state.otp_stage = "reset"
                                        st.success("✅ Code verified! Now set a new password.")
                                        st.rerun()
                                    else:
                                        st.error(f"❌ {msg}")

                    elif st.session_state.otp_stage == "reset":
                        with st.form("otp_reset_form"):
                            npw = st.text_input("New Password (min 8 chars)", type="password").strip()
                            confirm_npw = st.text_input("Confirm New Password", type="password").strip()
                            st.markdown("<br>", unsafe_allow_html=True)
                            submitted = st.form_submit_button("Update Password →", use_container_width=True)
                            if submitted:
                                if not npw or not confirm_npw:
                                    st.error("⚠️ All fields are mandatory.")
                                elif len(npw) < 8:
                                    st.error("⚠️ Password must be at least 8 characters long.")
                                elif not is_valid_password(npw):
                                    st.error("❌ Password must contain at least one uppercase, one lowercase, one digit, and one special symbol.")
                                elif npw != confirm_npw:
                                    st.error("❌ Passwords do not match.")
                                else:
                                    with get_db() as c:
                                        history = c.execute("SELECT password_hash FROM password_history WHERE email=?", (st.session_state.reset_email,)).fetchall()
                                        if any(check_txt(npw, h[0]) for h in history):
                                            st.error("❌ You have used this password before. Please choose a different password.")
                                        else:
                                            new_hash = hash_txt(npw)
                                            c.execute("UPDATE users SET password_hash=? WHERE email=?", (new_hash, st.session_state.reset_email))
                                            c.execute("INSERT INTO password_history (email, password_hash, created_at) VALUES (?, ?, ?)",
                                                      (st.session_state.reset_email, new_hash, datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
                                            c.commit()
                                            # Verify
                                            verify_hash = c.execute("SELECT password_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()[0]
                                            if check_txt(npw, verify_hash):
                                                st.success("✅ Password updated successfully!")
                                                time.sleep(2)
                                                st.session_state.reset_email = None
                                                st.session_state.reset_mode = None
                                                st.session_state.otp_stage = None
                                                st.session_state.otp_jwt = None
                                                navigate("Login")
                                            else:
                                                st.error("❌ Password update failed. Please try again.")

                st.markdown("<br>", unsafe_allow_html=True)
                if st.button("← Cancel", use_container_width=True):
                    st.session_state.reset_email = None
                    st.session_state.reset_mode = None
                    st.session_state.otp_stage = None
                    st.session_state.otp_jwt = None
                    st.session_state.sq_p = None
                    navigate("Login")

            elif st.session_state.page == "AdminLogin":
                admin_login_page()

    else:
        # ─── USER LOGGED IN ─────────────────────────────────────
        payload = verify_jwt(st.session_state.token)
        if not payload:
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

        email = payload["email"]
        with get_db() as c:
            uname = c.execute("SELECT username FROM users WHERE email=?", (email,)).fetchone()[0]

        with st.sidebar:
            st.markdown(f"""
            <div style="padding:16px 8px;text-align:center;">

                <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">Infosys Franchise Analytics &amp; Management</div>
                <div style="font-size:11px;color:{COLORS['text_muted']};">{"Admin Panel" if email=="franchise.admin@infosys.com" else "Franchise Analytics"}</div>
            </div><hr style="border-color:{COLORS['border_light']};">
            """, unsafe_allow_html=True)

            opts = ["Dashboard", "Settings", "Logout"] if email == "franchise.admin@infosys.com" else ["Dashboard", "Analytics", "Reports", "Logout"]
            icons = ["house", "gear", "box-arrow-right"] if email == "franchise.admin@infosys.com" else ["house", "graph-up", "file-text", "box-arrow-right"]
            menu = option_menu(None, opts, icons=icons,
                               styles={
                                   "container": {"background-color": COLORS['bg_sidebar']},
                                   "nav-link-selected": {"background-color": COLORS['accent'], "color": COLORS['accent_text']}
                               })
            if menu == "Logout":
                st.session_state.token = None
                st.session_state.page = "Login"
                st.rerun()

        if email == "franchise.admin@infosys.com":
            st.markdown(f"""
            <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
                <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Franchise Analytics &amp; Management</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Franchise Analytics</div></div>
                <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};white-space:nowrap;">👤 {uname}</div>
            </div>
            """, unsafe_allow_html=True)

            st.markdown("### Registered Users")
            with get_db() as c:
                users = c.execute("SELECT username, email FROM users ORDER BY id").fetchall()
            if users:
                data = [{"Username": u[0], "Email": u[1]} for u in users]
                st.table(data)
            else:
                st.info("No users registered yet.")
        else:
            st.markdown(f"""
            <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
                <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Franchise Analytics & Management</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Franchise Analytics</div></div>
                <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">👤 {uname}</div>
            </div>
            """, unsafe_allow_html=True)

            c1, c2, c3, c4 = st.columns(4)
            for col, icon, lbl, val in [
                (c1, "📄", "Documents Indexed", "128"),
                (c2, "🔍", "Searches Today", "47"),
                (c3, "📊", "Efficiency Score", "98.4%"),
                (c4, "🛡️", "Security Status", "Secured")
            ]:
                col.markdown(f"""
                <div class="pn-card" style="text-align:center;">
                    <div style="font-size:28px;">{icon}</div>
                    <div style="font-size:26px;font-weight:700;color:{COLORS['text_heading']};">{val}</div>
                    <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">{lbl}</div>
                </div>
                """, unsafe_allow_html=True)

            st.markdown("<br>", unsafe_allow_html=True)
            fig = go.Figure(go.Indicator(
                mode="gauge+number",
                value=92,
                title={"text": "System Health Index", "font": {"color": COLORS['text_heading'], "size": 14}},
                gauge={
                    "axis": {"range": [0, 100]},
                    "bar": {"color": COLORS['accent']},
                    "bgcolor": COLORS['bg_card_alt'],
                    "borderwidth": 1,
                    "bordercolor": COLORS['border']
                }
            ))
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"},
                              height=260, margin=dict(l=10, r=10, t=40, b=10))
            st.plotly_chart(fig, use_container_width=True)

In [ ]:
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# ────────────────────── 1 ────────────────────────
os.environ['JWT_SECRET'] = userdata.get('JWT_SECRET')
os.environ['EMAIL_ADDRESS'] = userdata.get('EMAIL_ADDRESS')
os.environ['EMAIL_PASSWORD'] = userdata.get('EMAIL_PASSWORD')
os.environ['NGROK_AUTHTOKEN'] = userdata.get('NGROK_AUTHTOKEN')

# ────────────────────── 2 ────────────────────────
ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])

# ────────────────────── 3 ────────────────────────
ngrok.kill()
!pkill -f streamlit
time.sleep(2)

# ────────────────────── 4 ────────────────────────
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=os.environ.copy()
)

# ────────────────────── 5 ────────────────────────
time.sleep(5)
public_url = ngrok.connect(8501).public_url
print("=" * 60)
print(f"🚀 Infosys Portal Live URL: {public_url}")
print("=" * 60)
print("⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")

# ────────────────────── 6 ────────────────────────
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Shutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("✅ Clean exit.")